In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import nltk
import re

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
df = pd.read_csv('/content/customer_support_text_classification.csv')

df.head()

,ticket_id,channel,customer_message,sentiment_label,word_count,urgent_flag
0,TKT00001,chat,I need information about the payment process. ...,neutral,18,1
1,TKT00002,phone,I need information about the payment process.,neutral,7,0
2,TKT00003,email,The refund process was fast and convenient. I ...,positive,12,0
3,TKT00004,social,My refund is still pending and this experience...,negative,15,1
4,TKT00005,chat,Please tell me how to update my account details.,neutral,9,0


In [11]:
print("Number of Records:", len(df))

print("\nColumns:")
print(df.columns)

print("\nSample Records:")
print(df[['customer_message', 'sentiment_label']].head())

print("\nClass Distribution:")
print(df['sentiment_label'].value_counts())

Number of Records: 1500

Columns:
Index(['ticket_id', 'channel', 'customer_message', 'sentiment_label',
       'word_count', 'urgent_flag'],
      dtype='object')

Sample Records:
                                    customer_message sentiment_label
0  I need information about the payment process. ...         neutral
1      I need information about the payment process.         neutral
2  The refund process was fast and convenient. I ...        positive
3  My refund is still pending and this experience...        negative
4   Please tell me how to update my account details.         neutral

Class Distribution:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64


In [12]:
df['text_length'] = df['customer_message'].astype(str).apply(len)

print("Average Text Length:", df['text_length'].mean())

Average Text Length: 72.75666666666666


In [13]:
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

df['clean_text'] = df['customer_message'].astype(str).apply(clean_text)

df[['customer_message', 'clean_text']].head()

,customer_message,clean_text
0,I need information about the payment process. ...,need information payment process ticket number...
1,I need information about the payment process.,need information payment process
2,The refund process was fast and convenient. I ...,refund process fast convenient appreciate quic...
3,My refund is still pending and this experience...,refund still pending experience frustrating ti...
4,Please tell me how to update my account details.,please tell update account details


In [14]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_text'])

y = df['sentiment_label']

print(X.shape)

(1500, 146)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)

print("Testing Shape:", X_test.shape)

Training Shape: (1200, 146)
Testing Shape: (300, 146)


In [16]:
model = LogisticRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")

print(classification_report(y_test, predictions))

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [17]:
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': predictions
})

results.to_csv('model_evaluation.csv', index=False)

results.head()

,Actual,Predicted
1116,neutral,neutral
1368,positive,positive
422,neutral,neutral
413,negative,negative
451,negative,negative


In [18]:
sample_predictions = results.head(10)

with open("sample_predictions.txt", "w") as f:
    f.write(sample_predictions.to_string())

print("Saved Successfully")

Saved Successfully


# Task 5: Sequence Model Architecture

## Input Sequence
The customer messages are converted into sequences of tokens.

## Embedding Layer
The embedding layer transforms words into dense numerical vectors.

## LSTM Layer
The LSTM processes words sequentially and remembers important contextual information.

## Output Layer
The dense output layer predicts:
- positive
- neutral
- negative

## Loss Function
Categorical Crossentropy

## Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1-score

# Task 6: Attention and Transformer Reflection

## Why RNNs Struggle
RNNs struggle with long-term dependencies because information from earlier words can vanish during long sequences.

## How LSTMs Help
LSTMs use memory cells and gates to preserve important information for longer periods.

## What Attention Solves
Attention mechanisms help models focus on important words in the sequence.

## Why Transformers are Important
Transformers process text efficiently in parallel and are widely used in modern NLP and Generative AI systems such as ChatGPT.